## Supplementary Table 3-5

Enviorment: env 1

In [1]:
import pandas as pd
import pickle
import os
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix
import numpy as np

### load the data

In [2]:
# for the meta pkls
# 0: prediction
# 1: probs
# 2: f1
# 3: feature sets
# 4: ground truth labels
# 5: model params

In [3]:
pw1_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw1/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw2_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw2/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw3_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw3/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw4_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw4/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_8}_oof.pkl"
pw5_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw5/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_0.8,scale_pos_weight_12,n_feats_32}_oof.pkl"
pw6_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw6/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_8}_oof.pkl"
pw7_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw7/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_1,learning_rate_0.1,subsample_0.8,scale_pos_weight_2,n_feats_32}_oof.pkl"
pw8_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw8/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_0.8,scale_pos_weight_2,n_feats_8}_oof.pkl"
pw9_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw9/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_1.0,scale_pos_weight_12,n_feats_32}_oof.pkl"
pw10_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw10/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_1,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_32}_oof.pkl"

all_loo_paths = [pw1_5, pw2_5, pw3_5, pw4_5, pw5_5, pw6_5, pw7_5, pw8_5, pw9_5, pw10_5]

In [4]:
cu_all_probs = []
for i in range(10):
    full_path = all_loo_paths[i]
    with open(full_path, "rb") as f:
        data = pickle.load(f)
    cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [5]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

In [6]:
fmf_loo = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_cu.csv")

In [7]:
fmf_loo["fmf_prob"] = fmf_loo["pe_before_delivery_percent"] / 100
fmf_loo["id"] = [id.lower() for id in fmf_loo["patient_id"]]
fmf_loo = fmf_loo[["fmf_prob", "id"]]

In [8]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")
cu_all_probs = cu_all_probs.merge(fmf_loo, how="left")

In [9]:
# number subjects with predictions
len(cu_all_probs[~cu_all_probs[0].isna() | ~cu_all_probs[1].isna() | ~cu_all_probs[2].isna() | ~cu_all_probs[3].isna() | ~cu_all_probs[4].isna() | ~cu_all_probs[5].isna() |
    ~cu_all_probs[6].isna() | ~cu_all_probs[7].isna() | ~cu_all_probs[8].isna() | ~cu_all_probs[9].isna()])

1137

In [10]:
# number of pec cases with predictions
len(cu_all_probs[~cu_all_probs[0].isna() & ~cu_all_probs[1].isna() & ~cu_all_probs[2].isna() & ~cu_all_probs[3].isna() & ~cu_all_probs[4].isna() & ~cu_all_probs[5].isna() &
    ~cu_all_probs[6].isna() & ~cu_all_probs[7].isna() & ~cu_all_probs[8].isna() & ~cu_all_probs[9].isna()]) - 82

54

In [11]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017.pkl", "rb") as f:
    all_pec = pickle.load(f)

In [12]:
cu_all_probs["label"] = [int(id in all_pec) for id in cu_all_probs["id"]]
sum(cu_all_probs["label"])

54

In [13]:
rename_dict = {}

for i in range(10):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [14]:
data_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data"

In [15]:
sf_cases = f"{data_path}/final_sf_20251028.pkl"
with open(sf_cases, "rb") as f:
    sf_cases = pickle.load(f)
nsf_cases = f"{data_path}/final_nsf_20251028.pkl"
with open(nsf_cases, "rb") as f:
    nsf_cases = pickle.load(f)
eope_cases = f"{data_path}/final_eope_20251017.pkl"
with open(eope_cases, "rb") as f:
    eope_cases = pickle.load(f)
lope_cases = f"{data_path}/final_lope_20251028.pkl"
with open(lope_cases, "rb") as f:
    lope_cases = pickle.load(f)
pec_cases = f"{data_path}/final_pec_20251017.pkl"
with open(pec_cases, "rb") as f:
    pec_cases = pickle.load(f)

In [16]:
pec_data = cu_all_probs[['pw_1', 'pw_2', 'pw_3', 'pw_4', 'pw_5', 'pw_6', 'pw_7', 'pw_8', 'pw_9', 'pw_10', 'label', 'id', "Past PEC", "First Preg", "Age at enrollment"]].copy()
pec_data.loc[:, "pec"] = [id in pec_cases for id in pec_data["id"]]
pec_data.loc[:, "sf"] = [id in sf_cases for id in pec_data["id"]]
pec_data.loc[:, "nsf"] = [id in nsf_cases for id in pec_data["id"]]
pec_data.loc[:, "eope"] = [id in eope_cases for id in pec_data["id"]]
pec_data.loc[:, "lope"] = [id in lope_cases for id in pec_data["id"]]

In [17]:
best_pp = 0.35
best_fp = 0.15
best_age = 37
reweight = cu_all_probs.copy()
reweight["Combo"] = (reweight["Past PEC"] * best_pp + best_fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > best_age).astype(int)))

In [18]:
for i in range(1, 11):
    pec_data[f"pw_{i}"] = pec_data[f"pw_{i}"] + (reweight["Past PEC"] * best_pp + best_fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > best_age).astype(int)))

### make the tables

In [19]:
cases = ["pec", "sf", "nsf", "eope", "lope"]
pw_results = None
thresh = 0.44
print(thresh)
for c in cases:
    for pw in range(1, 11):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column]))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        f1 = f1_score(grounds, predredictions)
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        auc_stat = roc_auc_score(grounds, probabilities)
        metric_df = pd.DataFrame({
            "cases": [c], "AUC": auc_stat, "PPV": ppv, "NPV": npv, "TPR": tpr, "FPR": fpr,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04), "F1": f1, "TP": tp, "TN": tn, "FP": fp, "FN": fn
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results["order"] = [3, 4, 2, 0, 1]
pw_results = pw_results.sort_values("order")
pw_results[["TP", "FP", "TN", "FN", "TPR", "FPR", "PPV", "NPV", "F1"]].T.rename(columns={"sf": "SF", "nsf": "NSF", "eope": "EOPE", "lope": "LOPE", "pec": "PEC"})[["SF", "NSF", "EOPE", "LOPE", "PEC"]].round(4)

0.44


cases,SF,NSF,EOPE,LOPE,PEC
TP,30.6000,13.1000,17.3000,26.4000,43.7000
FP,17.4000,17.4000,17.4000,17.4000,17.4000
TN,164.7000,164.7000,164.7000,164.7000,164.7000
FN,7.4000,2.9000,2.7000,7.6000,10.3000
TPR,0.8053,0.8188,0.8650,0.7765,0.8093
FPR,0.0956,0.0956,0.0956,0.0956,0.0956
PPV,0.6504,0.4580,0.5221,0.6201,0.7235
NPV,0.9567,0.9827,0.9836,0.9557,0.9409
F1,0.7170,0.5788,0.6459,0.6853,0.7622


In [20]:
cases = ["pec", "sf", "nsf", "eope", "lope"]
pw_results = None
thresh = 0.43
print(thresh)
for c in cases:
    for pw in range(1, 11):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column]))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        f1 = f1_score(grounds, predredictions)
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        auc_stat = roc_auc_score(grounds, probabilities)
        metric_df = pd.DataFrame({
            "cases": [c], "AUC": auc_stat, "PPV": ppv, "NPV": npv, "TPR": tpr, "FPR": fpr,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04), "F1": f1, "TP": tp, "TN": tn, "FP": fp, "FN": fn
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results["order"] = [3, 4, 2, 0, 1]
pw_results = pw_results.sort_values("order")
pw_results = pw_results[["TP", "FP", "TN", "FN", "TPR", "FPR", "PPV", "NPV", "F1"]].T.rename(columns={"sf": "SF", "nsf": "NSF", "eope": "EOPE", "lope": "LOPE", "pec": "PEC"})[["SF", "NSF", "EOPE", "LOPE", "PEC"]]
display(pw_results)

0.43


cases,SF,NSF,EOPE,LOPE,PEC
TP,30.900000,13.100000,17.400000,26.600000,44.000000
FP,17.800000,17.800000,17.800000,17.800000,17.800000
TN,164.300000,164.300000,164.300000,164.300000,164.300000
FN,7.100000,2.900000,2.600000,7.400000,10.000000
TPR,0.813158,0.818750,0.870000,0.782353,0.814815
FPR,0.097760,0.097760,0.097760,0.097760,0.097760
PPV,0.649055,0.454179,0.519652,0.617919,0.721342
NPV,0.958228,0.982605,0.984147,0.956688,0.942292
F1,0.718986,0.575133,0.645004,0.685792,0.763309


In [21]:
cases = ["pec", "sf", "nsf", "eope", "lope"]
pw_results = None
thresh = 0.24
print(thresh)
for c in cases:
    for pw in range(1, 11):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column]))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        f1 = f1_score(grounds, predredictions)
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        auc_stat = roc_auc_score(grounds, probabilities)
        metric_df = pd.DataFrame({
            "cases": [c], "AUC": auc_stat, "PPV": ppv, "NPV": npv, "TPR": tpr, "FPR": fpr,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04), "F1": f1, "TP": tp, "TN": tn, "FP": fp, "FN": fn
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results["order"] = [3, 4, 2, 0, 1]
pw_results = pw_results.sort_values("order")
pw_results = pw_results[["TP", "FP", "TN", "FN", "TPR", "FPR", "PPV", "NPV", "F1"]].T.rename(columns={"sf": "SF", "nsf": "NSF", "eope": "EOPE", "lope": "LOPE", "pec": "PEC"})[["SF", "NSF", "EOPE", "LOPE", "PEC"]]
display(pw_results)

0.24


cases,SF,NSF,EOPE,LOPE,PEC
TP,33.100000,14.400000,18.100000,29.400000,47.500000
FP,36.000000,36.000000,36.000000,36.000000,36.000000
TN,146.100000,146.100000,146.100000,146.100000,146.100000
FN,4.900000,1.600000,1.900000,4.600000,6.500000
TPR,0.871053,0.900000,0.905000,0.864706,0.879630
FPR,0.197721,0.197721,0.197721,0.197721,0.197721
PPV,0.501251,0.312072,0.361752,0.473008,0.586311
NPV,0.966881,0.988984,0.986460,0.969276,0.956656
F1,0.630413,0.456262,0.510359,0.604568,0.699037


In [22]:
cases = ["pec", "sf", "nsf", "eope", "lope"]
pw_results = None
thresh = 0.16
print(thresh)
for c in cases:
    for pw in range(1, 11):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column]))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        f1 = f1_score(grounds, predredictions)
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        auc_stat = roc_auc_score(grounds, probabilities)
        metric_df = pd.DataFrame({
            "cases": [c], "AUC": auc_stat, "PPV": ppv, "NPV": npv, "TPR": tpr, "FPR": fpr,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04), "F1": f1, "TP": tp, "TN": tn, "FP": fp, "FN": fn
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results["order"] = [3, 4, 2, 0, 1]
pw_results = pw_results.sort_values("order")
pw_results = pw_results[["TP", "FP", "TN", "FN", "TPR", "FPR", "PPV", "NPV", "F1"]].T.rename(columns={"sf": "SF", "nsf": "NSF", "eope": "EOPE", "lope": "LOPE", "pec": "PEC"})[["SF", "NSF", "EOPE", "LOPE", "PEC"]]
display(pw_results)

0.16


cases,SF,NSF,EOPE,LOPE,PEC
TP,34.600000,14.500000,18.800000,30.300000,49.100000
FP,50.900000,50.900000,50.900000,50.900000,50.900000
TN,131.200000,131.200000,131.200000,131.200000,131.200000
FN,3.400000,1.500000,1.200000,3.700000,4.900000
TPR,0.910526,0.906250,0.940000,0.891176,0.909259
FPR,0.279556,0.279556,0.279556,0.279556,0.279556
PPV,0.428383,0.244476,0.294770,0.397374,0.511338
NPV,0.973124,0.988328,0.989910,0.971528,0.962184
F1,0.576483,0.379220,0.442767,0.542939,0.649201
